# Quantum Machine Learning

This notebook demonstrates quantum machine learning primitives:

- **Feature encoding**: mapping classical data into quantum states
- **Quantum kernels**: computing inner products in feature space
- **Variational classifiers**: parameterized circuits for classification

These building blocks connect classical ML workflows to quantum circuits
and resource estimation.

In [ ]:
from qdk_pythonic.domains.ml import (
    AngleEncoding,
    AmplitudeEncoding,
    QuantumKernel,
    VariationalClassifier,
)
from qdk_pythonic.domains.common import HardwareEfficientAnsatz

## Feature Encoding

`AngleEncoding` maps each feature x_i to a rotation Ry(x_i) on qubit i.
This uses one qubit per feature and produces a shallow circuit.

In [ ]:
encoding = AngleEncoding(n_features=4)
data = [0.1, 0.5, 0.9, 1.2]

enc_circuit = encoding.to_circuit(data)

print(f"Features: {encoding.n_features}")
print(f"Qubits: {enc_circuit.qubit_count()}")
print(f"Gates: {enc_circuit.total_gate_count()}")
print(f"Depth: {enc_circuit.depth()}")
print()
print(enc_circuit.draw())

## Quantum Kernel

The quantum kernel computes k(x, y) = |<phi(x)|phi(y)>|^2 using the
compute-uncompute method: apply the encoding for x, then the adjoint
of the encoding for y, then measure. If all qubits return |0>, the
states are identical.

In [ ]:
kernel = QuantumKernel(encoding=AngleEncoding(n_features=4))

x = [0.1, 0.5, 0.9, 1.2]
y = [0.3, 0.7, 0.8, 1.0]

kernel_circuit = kernel.to_circuit(x=x, y=y)

print(f"Kernel qubits: {kernel_circuit.qubit_count()}")
print(f"Kernel gates: {kernel_circuit.total_gate_count()}")
print(f"Kernel depth: {kernel_circuit.depth()}")
print()
print(kernel_circuit.draw())

## Variational Classifier

The `VariationalClassifier` combines feature encoding with a
`HardwareEfficientAnsatz` (alternating rotation and entangling layers)
and a measurement for classification output.

In [ ]:
enc = AngleEncoding(n_features=4)
ansatz = HardwareEfficientAnsatz(n_qubits=4, depth=2)

classifier = VariationalClassifier(encoding=enc, ansatz=ansatz)

# Use dummy parameter values
params = [0.1] * ansatz.num_parameters
data = [0.3, 0.7, 1.1, 0.5]

clf_circuit = classifier.to_circuit(data=data, params=params)

print(f"Ansatz parameters: {ansatz.num_parameters}")
print(f"Classifier qubits: {clf_circuit.qubit_count()}")
print(f"Classifier gates: {clf_circuit.total_gate_count()}")
print(f"Classifier depth: {clf_circuit.depth()}")
print()
print(clf_circuit.draw())

## Scaling with Features

As the number of features grows, both the encoding and ansatz circuits expand.
Angle encoding uses one qubit per feature; the hardware-efficient ansatz adds
rotation and entangling layers on top.

In [ ]:
print(f"{'Features':>10}  {'Qubits':>8}  {'Params':>8}  {'Gates':>8}  {'Depth':>8}")
print("-" * 50)

for n in [2, 4, 8, 16]:
    e = AngleEncoding(n_features=n)
    a = HardwareEfficientAnsatz(n_qubits=n, depth=2)
    clf = VariationalClassifier(encoding=e, ansatz=a)
    dummy_data = [0.5] * n
    dummy_params = [0.1] * a.num_parameters
    c = clf.to_circuit(data=dummy_data, params=dummy_params)
    print(f"{n:>10}  {c.qubit_count():>8}  {a.num_parameters:>8}  "
          f"{c.total_gate_count():>8}  {c.depth():>8}")